# Stochastic line-tension sweep — analysis & visualisation

This notebook reads the **archived simulation histories** written by
`stochastic_tension.py` (one compressed `outputs/<tau..._sigma...>/history.hf5`
per run) and produces everything that used to be baked into the run script:

* a colour-coded **GIF** (edges tinted by their live line tension) and **still
  frames** per run,
* two per-run metrics — the **degree of stochastic vertex movement** and the
  **number of T1 transitions** (neighbour exchanges),
* the sweep-level **summary figures** (movement / T1 heatmaps over the tau–sigma
  grid, plus a movement-vs-sigma line plot),
* a `metrics.csv`.

Because it only *reads* the archives, you can re-run the analysis as often as you
like without re-simulating — and re-run `stochastic_tension.py` without disturbing
any analysis you have already produced.

Run from the repo's **`vivarium-tyssue` conda env** (needs ImageMagick `magick`
on PATH for the GIFs).

In [ ]:
from __future__ import annotations
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Simulation config is the single source of truth — import it from the sim script.
import stochastic_tension as sim
from stochastic_tension import TAUS, SIGMAS, OUT_DIR, DATA_DIR, DATASET_NAME

# Put the repo root on the path so `vivarium_tyssue.draw` (used by the GIF helpers)
# and `tyssue` import cleanly regardless of where the kernel was launched.
sys.path.insert(0, str(sim.REPO))
from tyssue.core.history import HistoryHdf5

# --- visualisation / analysis constants (were in the old monolithic script) ---
COORDS = ["x", "y"]
EDGE_CMAP = "coolwarm"                # diverging: line tension is signed around 0
TAU_COLORS = ["#0072B2", "#E69F00", "#009E73", "#D55E00"]   # Okabe-Ito
NUM_GIF_FRAMES = 60
STILL_FRACTIONS = [0.0, 0.33, 0.66, 1.0]
FIG_DPI = 300
GIF_DPI = 110

print("tau grid   :", TAUS)
print("sigma grid :", SIGMAS)
print("archives   :", OUT_DIR)

## Loading an archived history

`stochastic_tension.py` writes each run with the same per-element keys tyssue's
`History.to_archive` uses, so we reopen it with `HistoryHdf5.from_archive` (giving
back `retrieve(t)` / `time_stamps` for drawing). The **full stacked** dataframes
(all timepoints) are read straight from the HDF5 store for the metrics. The small
`LoadedHistory` wrapper exposes exactly the `.datasets` / `.time_stamps` /
`.retrieve()` surface the analysis functions expect, so they are unchanged from the
original script.

In [ ]:
class LoadedHistory:
    # Reopened archive presenting the same surface as an in-memory tyssue History:
    # .time_stamps, .retrieve(t) (drawing) and .datasets (full stacked dfs, analysis).
    def __init__(self, path):
        self._h = HistoryHdf5.from_archive(str(path))
        self._sheet = self._h.sheet
        with pd.HDFStore(str(path), "r") as store:
            self.datasets = {k.strip("/"): store.select(k) for k in store.keys()}
        self._times = np.array(sorted(self.datasets["vert"]["time"].unique()))

    @property
    def time_stamps(self):
        return self._times

    def retrieve(self, t):
        # Rebuild a sheet at the nearest recorded time from the stacked datasets.
        # We restore each element's per-frame topological index from its `vert` /
        # `edge` / `face` column (HistoryHdf5.retrieve skips this, leaving the stacked
        # RangeIndex so srce/trgt no longer line up with vert_df and `.loc` /
        # SheetGeometry.update_all break). This mirrors in-memory History.retrieve.
        t = self._times[int(np.argmin(np.abs(self._times - t)))]
        sheet_datasets = {}
        for elem, df in self.datasets.items():
            sub = df[df["time"] == t]
            if elem in sub.columns:
                sub = sub.set_index(elem)
                sub.index.name = elem
            sheet_datasets[elem] = sub
        sheet = type(self._sheet)(f"{self._sheet.identifier}_{t:04.3f}",
                                  sheet_datasets, self._sheet.specs)
        sheet.coords = self._sheet.coords
        return sheet

    def update_datasets(self):
        pass  # already stacked at load time

    def __getattr__(self, name):
        # Delegate anything else (e.g. `.sheet`, used by tyssue.create_gif) to the
        # underlying reloaded HistoryHdf5.
        return getattr(self._h, name)


def archive_path(tau, sigma) -> Path:
    return OUT_DIR / f"tau{tau:0.2f}_sigma{sigma:0.2f}" / "history.hf5"

## Visualisation — colour-coded GIF + stills

Edges are tinted by their live `line_tension` (diverging `coolwarm`, scaled to
±3σ so hue is comparable within a run). Identical to the old `save_gif` /
`save_stills`, just reading a reloaded history.

In [ ]:
def _edge_rgba(edge_df, color_range):
    # (Ne, 4) RGBA colouring each edge by its line_tension.
    cmap = plt.get_cmap(EDGE_CMAP)
    cmin, cmax = color_range
    vals = edge_df["line_tension"].to_numpy().astype(float)
    normed = np.clip((vals - cmin) / (cmax - cmin), 0.0, 1.0)
    return cmap(normed)


def save_gif(history, out_path: Path, sigma: float):
    # Colour-coded animation: edges tinted by their live line tension.
    from tyssue import config
    from tyssue.draw import create_gif
    from vivarium_tyssue.draw import line_tension_edge_kwds

    crange = (-3.0 * sigma, 3.0 * sigma)
    draw_specs = config.draw.sheet_spec()
    draw_specs["face"]["visible"] = True
    draw_specs["face"]["color"] = "#dddddd"
    draw_specs["face"]["alpha"] = 0.6
    draw_specs["vert"]["visible"] = False
    draw_specs.update(line_tension_edge_kwds(color_range=crange, colormap=EDGE_CMAP, width=1.2))
    create_gif(history, str(out_path), coords=COORDS, num_frames=NUM_GIF_FRAMES, dpi=GIF_DPI, **draw_specs)


def save_stills(history, out_dir: Path, sigma: float):
    # Save still frames coloured by line tension at a few timepoints.
    from tyssue.draw import sheet_view
    from tyssue.geometry.sheet_geometry import SheetGeometry

    crange = (-3.0 * sigma, 3.0 * sigma)
    times = list(history.time_stamps)
    if not times:
        return
    for frac in STILL_FRACTIONS:
        t = times[int(round(frac * (len(times) - 1)))]
        sheet = history.retrieve(t)
        SheetGeometry.update_all(sheet)
        fig, ax = plt.subplots(figsize=(4.0, 4.0))
        sheet_view(
            sheet, coords=COORDS, ax=ax,
            face={"visible": True, "color": "#dddddd", "alpha": 0.6},
            edge={"visible": True, "color": _edge_rgba(sheet.edge_df, crange), "width": 1.2},
        )
        ax.set_title(f"t = {float(t):.1f}", fontsize=9)
        ax.set_aspect("equal")
        fig.savefig(out_dir / f"still_t{float(t):04.1f}.png", dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)

## Analysis — vertex movement & T1 transitions

Both metrics match vertices/faces by `unique_id` across frames (identity is stable
across reconnections; the positional index is not). Unchanged from the original
script.

In [ ]:
def vertex_movement(history):
    # Degree of stochastic vertex movement. Returns (mean_step_rms, mean_path_length):
    # RMS per-step vertex displacement averaged over the run, and total distance
    # travelled per vertex averaged over vertices.
    times = list(history.time_stamps)
    prev = None
    per_step_rms = []
    path = defaultdict(float)
    for t in times:
        sheet = history.retrieve(t)
        vdf = sheet.vert_df
        uids = vdf["unique_id"].to_numpy()
        pos = vdf[COORDS].to_numpy(dtype=float)
        cur = {int(u): pos[i] for i, u in enumerate(uids)}
        if prev is not None:
            common = sorted(set(cur) & set(prev))
            if common:
                deltas = np.array([np.linalg.norm(cur[u] - prev[u]) for u in common])
                per_step_rms.append(float(np.sqrt(np.mean(deltas ** 2))))
                for u, d in zip(common, deltas):
                    path[u] += float(d)
        prev = cur
    mean_step_rms = float(np.mean(per_step_rms)) if per_step_rms else 0.0
    mean_path_length = float(np.mean(list(path.values()))) if path else 0.0
    return mean_step_rms, mean_path_length


def _face_adjacency(sheet):
    # Set of neighbouring face pairs (share >= 2 vertices), keyed by face unique_id.
    edf = sheet.edge_df
    vuid = sheet.vert_df["unique_id"].to_numpy()
    fuid = sheet.face_df["unique_id"].to_numpy()

    face_vsets = defaultdict(set)
    for f, s in zip(edf["face"].to_numpy(), edf["srce"].to_numpy()):
        face_vsets[int(f)].add(int(vuid[int(s)]))

    vert_faces = defaultdict(set)
    for fpos, vset in face_vsets.items():
        for v in vset:
            vert_faces[v].add(fpos)

    shared = defaultdict(int)
    for faces in vert_faces.values():
        fl = sorted(faces)
        for i in range(len(fl)):
            for j in range(i + 1, len(fl)):
                shared[(fl[i], fl[j])] += 1

    adj = set()
    for (a, b), n in shared.items():
        if n >= 2:
            adj.add(frozenset((int(fuid[a]), int(fuid[b]))))
    return adj


def count_t1_transitions(history):
    # Estimate the number of T1 transitions (neighbour exchanges) over the run, as
    # sum of len(symmetric_difference)/2 of the face-adjacency set between frames.
    times = list(history.time_stamps)
    prev = None
    total = 0.0
    for t in times:
        adj = _face_adjacency(history.retrieve(t))
        if prev is not None:
            total += len(prev ^ adj) / 2.0
        prev = adj
    return float(total)

## Sweep summary figures

In [ ]:
def _heatmap(ax, grid, title, cbar_label):
    im = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(SIGMAS)), [f"{s:g}" for s in SIGMAS])
    ax.set_yticks(range(len(TAUS)), [f"{t:g}" for t in TAUS])
    ax.set_xlabel("sigma (noise amplitude)")
    ax.set_ylabel("tau (relaxation time)")
    ax.set_title(title, fontsize=11)
    thresh = np.nanmin(grid) + 0.6 * (np.nanmax(grid) - np.nanmin(grid))
    for i in range(len(TAUS)):
        for j in range(len(SIGMAS)):
            v = grid[i, j]
            ax.text(j, i, f"{v:.2g}", ha="center", va="center", fontsize=8,
                    color="white" if v < thresh else "black")
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(cbar_label, fontsize=9)


def build_summary(df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    move = df.pivot(index="tau", columns="sigma", values="mean_step_rms").reindex(index=TAUS, columns=SIGMAS)
    t1 = df.pivot(index="tau", columns="sigma", values="t1_count").reindex(index=TAUS, columns=SIGMAS)

    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    _heatmap(ax, move.to_numpy(), "Stochastic vertex movement", "mean per-step RMS displacement")
    fig.tight_layout(); fig.savefig(out_dir / "movement_heatmap.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    _heatmap(ax, t1.to_numpy(), "T1 transitions (neighbour exchanges)", "estimated T1 count")
    fig.tight_layout(); fig.savefig(out_dir / "t1_heatmap.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    for k, tau in enumerate(TAUS):
        sub = df[df["tau"] == tau].sort_values("sigma")
        ax.plot(sub["sigma"], sub["mean_step_rms"], "-o", color=TAU_COLORS[k % len(TAU_COLORS)],
                markersize=5, linewidth=2, label=f"tau = {tau:g}")
    ax.set_xlabel("sigma (noise amplitude)")
    ax.set_ylabel("mean per-step RMS displacement")
    ax.set_title("Vertex movement vs noise amplitude", fontsize=11)
    ax.legend(frameon=False, fontsize=9)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    fig.tight_layout(); fig.savefig(out_dir / "metric_vs_sigma.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

## Run the analysis over the whole sweep

For every `(tau, sigma)` whose `history.hf5` exists, this renders the GIF + stills
and computes the two metrics, then builds `metrics.csv` and the summary figures.
Set `MAKE_GIFS = False` to skip the (slow) animation rendering while iterating on
the metrics/plots.

In [ ]:
MAKE_GIFS = True

rows = []
for tau in TAUS:
    for sigma in SIGMAS:
        path = archive_path(tau, sigma)
        if not path.exists():
            print(f"[skip] {path.parent.name}: no history.hf5 (run stochastic_tension.py first)")
            continue
        run_dir = path.parent
        print(f"[run] {run_dir.name} ...", flush=True)
        history = LoadedHistory(path)

        if MAKE_GIFS:
            save_gif(history, run_dir / "stochastic.gif", sigma)
        save_stills(history, run_dir, sigma)

        mean_step_rms, mean_path_length = vertex_movement(history)
        t1_count = count_t1_transitions(history)
        print(f"       movement(rms)={mean_step_rms:.4g} path={mean_path_length:.4g} T1={t1_count:.0f}")
        rows.append({
            "tau": tau, "sigma": sigma,
            "mean_step_rms": mean_step_rms,
            "mean_path_length": mean_path_length,
            "t1_count": t1_count,
        })

df = pd.DataFrame(rows)
df

In [ ]:
if not df.empty:
    df.to_csv(OUT_DIR / "metrics.csv", index=False)
    print("wrote", OUT_DIR / "metrics.csv")
    build_summary(df, OUT_DIR / "summary")
else:
    print("No archives found — run `python stochastic_tension.py` first.")